In [ ]:
!pip install transformers datasets accelerate evaluate -q

In [ ]:
!pip install transformers datasets evaluate accelerate -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset

In [ ]:
df = pd.read_csv("/content/sentiment_real_reviews_1000.csv")

df = df[["reviews", "sentiment_label"]].dropna()
df = df.rename(columns={"reviews": "text", "sentiment_label": "label"})

print(df.head())
print(df["label"].value_counts())

In [ ]:
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

df["label"] = df["label"].map(label2id)

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_sentiment",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch", # Added this line to match eval_strategy
    learning_rate=2e-5,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results=trainer.evaluate()
print(results)

In [ ]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))
print(confusion_matrix(y_true, y_pred))

In [ ]:
import torch

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return id2label[predicted_class]

sample_reviews = [
    "This product is excellent and works perfectly. I am very happy with it.",
    "The item is okay. Not bad, but nothing special.",
    "Very disappointing product. It stopped working after two days."
]

for review in sample_reviews:
    print("Review:", review)
    print("Predicted sentiment:", predict_sentiment(review))
    print("-" * 60)

In [ ]:
model.save_pretrained("/content/bert_sarcasm_model")
tokenizer.save_pretrained("/content/bert_sarcasm_model")

Testing Zero-shot model for sentiment analysis

In [ ]:
!pip install transformers -q

from transformers import pipeline
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

In [ ]:
df = pd.read_csv("/content/sentiment_real_reviews_1000.csv")

df = df[["reviews", "sentiment_label"]].dropna()

print(df.head())

In [ ]:
labels = ["positive", "neutral", "negative"]

In [ ]:
sample_reviews = [
    "This product is excellent and works perfectly.",
    "The product is okay, nothing special.",
    "Very disappointing item."
]

for r in sample_reviews:
    result = classifier(r, labels)
    print("Review:", r)
    print("Prediction:", result["labels"][0])
    print("Scores:", dict(zip(result["labels"], result["scores"])))
    print("-"*60)

In [ ]:
predictions = []

for text in df["reviews"]:
    result = classifier(text, labels)
    predictions.append(result["labels"][0])

In [ ]:
print("Accuracy:", accuracy_score(df["sentiment_label"], predictions))

print("\nClassification Report:\n")
print(classification_report(df["sentiment_label"], predictions))